<a href="https://colab.research.google.com/github/sidatKS/Anthropic_CPN/blob/main/Model_grading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install anthropic

In [24]:
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
from anthropic import Anthropic
client = Anthropic()  # auto-picks up the env var
model = "claude-haiku-4-5"

In [25]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [26]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [27]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [28]:
def run_prompt(test_case):
  """Merges the prompt and tasks from dataset, then returns the result"""
  prompt = f"""
Please solve the following task:
{test_case['task']}
"""
  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```python")
  output = chat(messages, stop_sequences=["```"])
  return output, prompt

In [29]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.

    Task: {test_case['task']}
    Solution: {output}

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [30]:
def run_test_case(test_case):
  """calls run_prompt, then grades the results"""
  output, prompt = run_prompt(test_case)
  #TODO GRADING
  model_grade = grade_by_model(test_case, output)
  score = model_grade["score"]
  reasoning = model_grade["reasoning"]
  return{
      "test_case": test_case,
      "prompt": prompt,
      "output": output,
      "reasoning": reasoning,
      "score": score

  }

In [33]:
def run_eval(dataset):
  """Loads the dataset and calls run_test_case with each case"""
  all_results = []
  for test_case in dataset:
    result = run_test_case(test_case)
    all_results.append(result)
  return all_results

In [37]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)

In [38]:
print(json.dumps(results, indent=2))

[
  {
    "test_case": {
      "task": "Write a Python function that parses an AWS S3 bucket URI (e.g., 's3://my-bucket/path/to/file.txt') and returns a dictionary with 'bucket' and 'key' as separate fields."
    },
    "prompt": "\nPlease solve the following task:\nWrite a Python function that parses an AWS S3 bucket URI (e.g., 's3://my-bucket/path/to/file.txt') and returns a dictionary with 'bucket' and 'key' as separate fields.\n",
    "output": "\ndef parse_s3_uri(uri: str) -> dict:\n    \"\"\"\n    Parses an AWS S3 bucket URI and returns bucket and key as separate fields.\n    \n    Args:\n        uri (str): S3 URI in the format 's3://bucket-name/path/to/file'\n        \n    Returns:\n        dict: Dictionary with 'bucket' and 'key' fields\n        \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n        \n    Examples:\n        >>> parse_s3_uri('s3://my-bucket/path/to/file.txt')\n        {'bucket': 'my-bucket', 'key': 'path/to/file.txt'}\n        \n        >>>